# sat_arl — Satélite ARL

Este notebook construye el **satélite ARL**, que consolida la información de personas
registradas en el sistema AS400, unificando tres roles (Tomador, Asegurado, Beneficiario)
en una sola tabla.

## Tablas fuente

| Tabla | Qué contiene |
|---|---|
| `AAAFAAF0` | Datos del Tomador (nombre completo, documento, contacto, dirección) |
| `AAAFNAF0` | Contrato (AFNAFI, TIPO_CONTRATO, ATDP del tomador) |
| `AAICOAF0` | Clave asesor (ICOCLA, relacionada por ICOAFI → contrato) |
| `AAEMPAF0` | Datos del Asegurado (nombre, documento, contacto, dirección) |
| `AACIUAF0` | Catálogo de ciudades (CIUCDP = departamento, CIUCCI = código ciudad) |

## Tabla destino

`axa_col_slv_dv.stg_cliente.sat_arl`

## Ejecución — Crear el satélite ARL

La siguiente celda crea (o reemplaza) la tabla `axa_col_slv_dv.stg_cliente.sat_arl`.
Se aplica deduplicación por `fecha_cargue` en cada tabla fuente antes de los JOINs,
siguiendo el mismo patrón de las queries de referencia de SQL Server.

> ⚠️ **Importante:** Este proceso borra y recrea la tabla completa cada vez que se ejecuta.

In [ ]:
%sql

CREATE OR REPLACE TABLE axa_col_slv_dv.stg_cliente.sat_arl
USING DELTA AS

WITH

-- ── Deduplicación por fecha_cargue ─────────────────────────────────────────

afaaf AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY AFAAFI, AFANIT
                   ORDER BY fecha_cargue DESC
               ) AS rn
        FROM axa_col_slv_dv.core_as400.AAAFAAF0
    ) t
    WHERE rn = 1
),

afnaf AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY AFNAFI
                   ORDER BY fecha_cargue DESC
               ) AS rn
        FROM axa_col_slv_dv.core_as400.AAAFNAF0
    ) t
    WHERE rn = 1
),

icoaf AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY ICOAFI
                   ORDER BY fecha_cargue DESC
               ) AS rn
        FROM axa_col_slv_dv.core_as400.AAICOAF0
    ) t
    WHERE rn = 1
),

empaf AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY EMPIDE, EMPAFI, EMPSCO
                   ORDER BY fecha_cargue DESC
               ) AS rn
        FROM axa_col_slv_dv.core_as400.AAEMPAF0
    ) t
    WHERE rn = 1
),

ciudad AS (
    SELECT DISTINCT CIUCCI, CIUCDP
    FROM axa_col_slv_dv.core_as400.AACIUAF0
),

-- ── Rol: TOMADOR ────────────────────────────────────────────────────────────
-- Fuente principal: AAAFAAF0
-- Llaves: AFAAFI + AFANIT
-- Nota: el nombre no viene parseado; todo está en AFARSO (nombre completo)

tomador AS (
    SELECT
        -- Llaves / relaciones
        a.AFAAFI                                                    AS afaafi,
        a.AFANIT                                                    AS afanit,
        n.AFNAFI                                                    AS afnafi,
        -- Identificación
        a.AFATDO                                                    AS tipo_documento,
        a.AFANIT                                                    AS numero_documento,
        -- Nombre (no parseado en AS400; se registra solo el nombre completo)
        NULL                                                        AS primer_nombre,
        NULL                                                        AS primer_apellido,
        NULL                                                        AS segundo_nombre,
        NULL                                                        AS segundo_apellido,
        a.AFARSO                                                    AS nombre_completo_razon_social,
        -- Contacto
        a.AFAEML                                                    AS correo,
        COALESCE(NULLIF(TRIM(a.AFATE1), ''),
                 NULLIF(TRIM(a.AFATE2), ''))                        AS celular,
        -- Ubicación
        a.AFADIR                                                    AS direccion_residencial,
        a.AFACIU                                                    AS ciudad_residencia,
        a.CIUCDP                                                    AS departamento,
        'Colombia'                                                  AS pais,
        -- Demográficos (no disponibles para tomador en AS400)
        NULL                                                        AS fecha_nacimiento,
        NULL                                                        AS genero,
        NULL                                                        AS estado_civil,
        -- Contrato y asesor
        n.AFNAFI                                                    AS contrato,
        n.TIPO_CONTRATO                                             AS tipo_contrato,
        n.AFNFU3                                                    AS ATDP,
        i.ICOCLA                                                    AS clave_asesor,
        -- Control
        a.fecha_cargue                                              AS FEC_ACTUALIZACION,
        'TOMADOR'                                                   AS rol
    FROM afaaf a
    LEFT JOIN afnaf n  ON n.AFNAFI  = a.AFAAFI
    LEFT JOIN icoaf i  ON i.ICOAFI  = n.AFNAFI
),

-- ── Rol: ASEGURADO ──────────────────────────────────────────────────────────
-- Fuente principal: AAEMPAF0
-- Llaves: EMPIDE + EMPAFI + EMPSCO
-- Departamento vía JOIN AACIUAF0 (EMPCIU = CIUCCI)

asegurado AS (
    SELECT
        -- Llaves / relaciones
        e.EMPIDE                                                    AS afaafi,
        e.EMPIDE                                                    AS afanit,
        e.EMPAFI                                                    AS afnafi,
        -- Identificación
        e.EMPTDO                                                    AS tipo_documento,
        e.EMPIDE                                                    AS numero_documento,
        -- Nombre
        e.EMPNO1                                                    AS primer_nombre,
        e.EMPAP1                                                    AS primer_apellido,
        e.EMPNO2                                                    AS segundo_nombre,
        e.EMPAP2                                                    AS segundo_apellido,
        TRIM(CONCAT_WS(' ', e.EMPNO1, e.EMPNO2,
                            e.EMPAP1, e.EMPAP2))                    AS nombre_completo_razon_social,
        -- Contacto
        e.EMPEML                                                    AS correo,
        e.EMPTEL                                                    AS celular,
        -- Ubicación
        e.EMPDIR                                                    AS direccion_residencial,
        e.EMPCIU                                                    AS ciudad_residencia,
        c.CIUCDP                                                    AS departamento,
        'Colombia'                                                  AS pais,
        -- Demográficos
        e.EMPFNA                                                    AS fecha_nacimiento,
        e.EMPSEX                                                    AS genero,
        e.EMPESC                                                    AS estado_civil,
        -- Contrato y asesor (no aplica directamente para asegurado)
        e.EMPAFI                                                    AS contrato,
        NULL                                                        AS tipo_contrato,
        e.EMPEPA                                                    AS ATDP,
        NULL                                                        AS clave_asesor,
        -- Control
        e.fecha_cargue                                              AS FEC_ACTUALIZACION,
        'ASEGURADO'                                                 AS rol
    FROM empaf e
    LEFT JOIN ciudad c ON c.CIUCCI = e.EMPCIU
),

-- ── Rol: BENEFICIARIO ───────────────────────────────────────────────────────
-- Los datos de beneficiario en AS400 son limitados (nombre completo + doc).
-- Los campos demográficos y de contacto se obtienen cruzando con AAEMPAF0
-- cuando el beneficiario también es asegurado en otra póliza.

beneficiario AS (
    SELECT
        -- Llaves / relaciones
        e.EMPIDE                                                    AS afaafi,
        e.EMPIDE                                                    AS afanit,
        e.EMPAFI                                                    AS afnafi,
        -- Identificación
        e.EMPTDO                                                    AS tipo_documento,
        e.EMPIDE                                                    AS numero_documento,
        -- Nombre
        e.EMPNO1                                                    AS primer_nombre,
        e.EMPAP1                                                    AS primer_apellido,
        e.EMPNO2                                                    AS segundo_nombre,
        e.EMPAP2                                                    AS segundo_apellido,
        TRIM(CONCAT_WS(' ', e.EMPNO1, e.EMPNO2,
                            e.EMPAP1, e.EMPAP2))                    AS nombre_completo_razon_social,
        -- Contacto
        e.EMPEML                                                    AS correo,
        e.EMPTEL                                                    AS celular,
        -- Ubicación
        e.EMPDIR                                                    AS direccion_residencial,
        e.EMPCIU                                                    AS ciudad_residencia,
        c.CIUCDP                                                    AS departamento,
        'Colombia'                                                  AS pais,
        -- Demográficos
        e.EMPFNA                                                    AS fecha_nacimiento,
        e.EMPSEX                                                    AS genero,
        e.EMPESC                                                    AS estado_civil,
        -- Contrato y asesor
        e.EMPAFI                                                    AS contrato,
        NULL                                                        AS tipo_contrato,
        e.EMPEPA                                                    AS ATDP,
        NULL                                                        AS clave_asesor,
        -- Control
        e.fecha_cargue                                              AS FEC_ACTUALIZACION,
        'BENEFICIARIO'                                              AS rol
    FROM empaf e
    LEFT JOIN ciudad c ON c.CIUCCI = e.EMPCIU
    WHERE e.EMPSCO = 'B'
)

SELECT * FROM tomador
UNION ALL
SELECT * FROM asegurado
UNION ALL
SELECT * FROM beneficiario

## Validación — Verificar el resultado

In [ ]:
%sql
-- Total de filas cargadas por rol
SELECT
    rol,
    COUNT(*) AS total_filas
FROM axa_col_slv_dv.stg_cliente.sat_arl
GROUP BY rol
ORDER BY rol

In [ ]:
%sql
-- Vista previa de los primeros 5 registros
SELECT
    rol,
    tipo_documento,
    numero_documento,
    primer_nombre,
    primer_apellido,
    nombre_completo_razon_social,
    correo,
    celular,
    ciudad_residencia,
    departamento,
    pais,
    contrato,
    tipo_contrato,
    clave_asesor,
    FEC_ACTUALIZACION
FROM axa_col_slv_dv.stg_cliente.sat_arl
LIMIT 5